In [ ]:
import pandas as pd
import numpy as np 
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA 
from sklearn.ensemble import GradientBoostingRegressor


df_train = pd.read_csv('/kaggle/input/pruebas-saber/training_pruebas.csv', low_memory=False)

df_test = pd.read_csv('/kaggle/input/test-saberprubas/test_pruebas.csv', low_memory=False)

In [ ]:
import pandas as pd

# Eliminamos columnas que no tienen tanto peso para el estudio
columns_to_drop = [
    'ESTU_TIPODOCUMENTO', 'ESTU_TIPODOCUMENTOSB11', 'PERIODO',
    'INST_NOMBRE_INSTITUCION', 'ESTU_PRIVADO_LIBERTAD', 'ESTU_ESTADOINVESTIGACION', 'ESTU_ESTUDIANTE',
    'ESTU_NIVEL_PRGM_ACADEMICO', 'INST_CARACTER_ACADEMICO', 'ESTU_NACIONALIDAD', 'ESTU_TITULOOBTENIDOBACHILLER'
]
df_train.drop(columns=columns_to_drop, axis=1, inplace=True)
df_test.drop(columns=columns_to_drop, axis=1, inplace=True)


test_consecutivo = df_test['ESTU_CONSECUTIVO']
df_train.drop('ESTU_CONSECUTIVO', axis=1, inplace=True)


missing_ratio = df_train.isnull().mean()
columns_to_drop = missing_ratio[missing_ratio > 0.5].index.tolist()
df_train.drop(columns=columns_to_drop, axis=1, inplace=True)
df_test.drop(columns=columns_to_drop, axis=1, inplace=True)

#Imputar valores faltantes en df_train
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    mean = df_train[col].mean()
    median = df_train[col].median()
    df_train[col].fillna(median if abs(mean - median) / mean > 0.1 else mean, inplace=True)

categorical_cols = df_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    mode = df_train[col].mode()[0]
    df_train[col].fillna(mode, inplace=True)

# mputar valores faltantes en df_test usando estadísticas de df_train
common_numeric_cols = numeric_cols.intersection(df_test.select_dtypes(include=['int64', 'float64']).columns)
common_categorical_cols = categorical_cols.intersection(df_test.select_dtypes(include=['object']).columns)

for col in common_numeric_cols:
    mean = df_train[col].mean()
    df_test[col].fillna(mean, inplace=True)

for col in common_categorical_cols:
    mode = df_train[col].mode()[0]
    df_test[col].fillna(mode, inplace=True)

#Crear columna de edad y eliminamos 'ESTU_FECHANACIMIENTO'
def calcular_edad(fecha_nacimiento):
    fecha_nacimiento = pd.to_datetime(fecha_nacimiento, errors='coerce', dayfirst=True)
    hoy = pd.Timestamp('now')
    cumple_ya_ocurrido = (hoy.month > fecha_nacimiento.dt.month) | (
        (hoy.month == fecha_nacimiento.dt.month) & (hoy.day >= fecha_nacimiento.dt.day))
    edad = hoy.year - fecha_nacimiento.dt.year - (~cumple_ya_ocurrido).astype(int)
    return edad

# Aplicar en df_train
df_train['ESTU_EDAD'] = calcular_edad(df_train['ESTU_FECHANACIMIENTO'])
df_train.drop('ESTU_FECHANACIMIENTO', axis=1, inplace=True)
median_age = df_train['ESTU_EDAD'].median()
df_train['ESTU_EDAD'].fillna(median_age, inplace=True)

# Aplicar en df_test
df_test['ESTU_EDAD'] = calcular_edad(df_test['ESTU_FECHANACIMIENTO'])
df_test.drop('ESTU_FECHANACIMIENTO', axis=1, inplace=True)
df_test['ESTU_EDAD'].fillna(median_age, inplace=True)



In [ ]:
# Definir X e y
X = df_train.drop('PUNT_GLOBAL', axis=1)
y = df_train['PUNT_GLOBAL']

X_test_real = df_test.copy()

# Aplicar One-Hot Encoding a X y X_test_real
X = pd.get_dummies(X, drop_first=True)
X_test_real = pd.get_dummies(X_test_real, drop_first=True)

# Alinear las columnas entre entrenamiento y test
X, X_test_real = X.align(X_test_real, join='left', axis=1, fill_value=0)

# Dividir X e y en conjuntos de entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar las variables numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test_real)

# Aplicar PCA para reducir dimensionalidad
pca = PCA(n_components=30) 
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)


def evaluar_knn(X_train, y_train, X_val, y_val, X_test, k):

    knn_model = KNeighborsRegressor(n_neighbors=k, algorithm='auto')  # 'auto' selecciona el mejor algoritmo
    
    # Entrenar el modelo
    knn_model.fit(X_train, y_train)
    
    # Evaluar el modelo en los conjuntos de entrenamiento, validación y prueba
    y_train_pred = knn_model.predict(X_train)
    y_val_pred = knn_model.predict(X_val)
    y_test_pred = knn_model.predict(X_test)

    # Calcular métricas de evaluación
    mse_train = mean_squared_error(y_train, y_train_pred)
    mse_val = mean_squared_error(y_val, y_val_pred)
    mape_train = mean_absolute_percentage_error(y_train, y_train_pred)
    mape_test = mean_absolute_percentage_error(y_val, y_val_pred)  # Para test

    resultados = {
        'k': k,
        'MSE train': mse_train,
        'MAPE train': mape_train,
        'MSE val': mse_val,
        'MAPE test': mape_test  
    }
    
    return resultados

#K a probar
valores_k = [5, 10, 20, 30]
resultados_tabla = []

# Evaluar el modelo para cada valor de k
for k in valores_k:
    resultados = evaluar_knn(X_train_pca, y_train, X_val_pca, y_val, X_test_pca, k)
    resultados_tabla.append(resultados)

df_resultados = pd.DataFrame(resultados_tabla)
print(df_resultados)

In [ ]:
#el mejor modelo KNN 
mejor_k = 20 

# Crear el modelo KNN con el mejor valor de k
knn_model = KNeighborsRegressor(n_neighbors=mejor_k, algorithm='auto')

# entrenamiento completo
knn_model.fit(X_train_pca, y_train)

# Hacer predicciones en el conjunto de test
y_test_pred = knn_model.predict(X_test_pca)

# Crear un DataFrame con las predicciones
predicciones_df = pd.DataFrame({
    'ESTU_CONSECUTIVO': df_test['ESTU_CONSECUTIVO'], 
    'PUNT_GLOBAL_PREDIC': y_test_pred
})


predicciones_df.to_csv('predicciones_knn.csv', index=False)



Modelo GBM

In [ ]:

gbm_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

# Entrenar el modelo en el conjunto de entrenamiento
gbm_model.fit(X_train_scaled, y_train)

# Realizar predicciones en el conjunto de entrenamiento y en el conjunto de prueba
y_train_pred = gbm_model.predict(X_train_scaled)
y_test_pred = gbm_model.predict(X_test_scaled)


mse_train = mean_squared_error(y_train, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train, y_train_pred)


print(f"MSE en entrenamiento: {mse_train}")
print(f"MAPE en entrenamiento: {mape_train}")

# Guardar las predicciones en un archivo CSV para enviarlas
df_submission = pd.DataFrame({
    'ESTU_CONSECUTIVO': test_consecutivo, 
    'PUNT_GLOBAL': y_test_pred
})

df_submission.to_csv('submission_gbm.csv', index=False)

print("Archivo 'submission_gbm.csv' generado con éxito.")